# <span style="font-weight:bold;color:blue;font-size:1.1em"> Day 19: Linear Regression</span>

Today's notebook will introduce how to conduct a linear regression from start to finish including a hypothesis for significant correlation and reporting out. The steps are as follows:

1. Scatter Plot -- verify linear relationship assumption is appropriate.
2. Array data -- the `linregress` function will not accept table data.
3. Linear Regression Test -- we run `stats.linregress` on the correct arrays.
4. Reporting Out

In [1]:
from datascience import *
%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')
import numpy as np
import seaborn as sns
import scipy.stats as stats
from scipy.stats import gaussian_kde

## Read in some CSV data files from the web.
star = Table.read_table('starbucks.csv').drop(5)
star_beer =Table.read_table('starbucks_vs_beer.csv')
top_movies = Table.read_table('top_movies_2017.csv')
pers = Table.read_table('http://faculty.ung.edu/rsinn/personality.csv')

In [2]:
def describe(data, var = None):
  if isinstance(data, Table):
    data = data.column(var)
  names = np.array(["mean", "std", "n", " : : ", "min", "Q1", "median", "Q3", "max", "IQR"])
  values = np.array([round(np.mean(data),1), round(stats.tstd(data),2), len(data) - 1, "", min(data), np.percentile(data, 25), np.median(data), np.percentile(data, 75), max(data), stats.iqr(data)])
  tab = Table().with_columns('Names', names, 'Values', values)  ;  transposed_tab = Table().with_columns( zip(tab.column('Names'), tab.column('Values')) ).show()
  return

def describes(data, num, group):
  dat = data.column(num)  ;  group_var = data.column(group)  ;  groups = np.unique(group_var)  ;  rows = []  ;  tab = Table(["mean", "std", "n", " : : ", "min", "Q1", "median", "Q3", "max", "IQR"])
  for g in groups:
    # Filter data for the current group
    sub_dat = dat[group_var == g]  ;    rows = [g, round(np.mean(sub_dat), 1), round(stats.tstd(sub_dat), 2), len(sub_dat) - 1, min(sub_dat), np.percentile(sub_dat, 25), np.median(sub_dat), np.percentile(sub_dat, 75), max(sub_dat), stats.iqr(sub_dat)]        
    tab = tab.with_row(rows)
  tab.show()
  return

def box(data, num = None, sm = False):
  if isinstance(data, Table):
    df = data.to_df()  ;  sns.boxplot(data=df, y=num, width=0.45, showmeans=sm)  ;
    plots.title(f'Boxplot of {num}')  ;  plots.xticks([], [])  ;  plots.show()  
  else:
    if num == True:
      sm = True  ;  num = 'Variable'
    if num == None:
      num = 'Variable'
    plots.figure(figsize=(6, 6))  ;  plots.boxplot(data, widths = 0.45, showmeans=sm)
    plots.title(f'Boxplot of {num}')  ;  plots.xticks([], [])  ;  plots.show()
  return

def boxes(data, num, cat, sm = False):
  if not isinstance(data, Table):
    print("ERROR: First input must be table name.")
  else:
    df = data.to_df()  ;  sns.boxplot(data=data, x=cat, y=num, width=0.75, showmeans=sm)
    plots.title(f'Boxplot of {num} Grouped by {cat}')  ;  plots.show()
  return

def density(data, num = None):
  if isinstance(data, Table):
    sns.kdeplot( data = data.column(num), bw_method="scott", bw_adjust=0.75, cut=3 )  ;  plots.title(f'Density Plot of {num}')  ;  plots.show()
  else:
    if num == None:
      num = 'Variable'  ;  sns.kdeplot( data = data, bw_method="scott", bw_adjust=0.75, cut=3 )  ;  plots.title(f'Density Plot of {num}')  ;  plots.show()
    else:
      sns.kdeplot( data = data, bw_method="scott", bw_adjust=0.75, cut=3 )  ;  plots.title(f'Density Plot of {num}')  ;  plots.show()
  return

def densities(tab, var, group_var):
  groups = tab.column(group_var) ;  unique_groups = np.unique(groups)  ;  plots.figure()
  for g in unique_groups:
    # Filter data for the current group
    mask = (groups == g)
    x = tab.column(var)[mask]   
    # Calculate KDE for the group subset
    kde = gaussian_kde(x)
    x_grid = np.linspace(x.min() - 1, x.max() + 1, 512)  ;    plots.plot(x_grid, kde(x_grid), label=str(g))   
  plots.title(f'Density Plot of {var} by {group_var}')  ;  plots.xlabel(f'{var}')  ;  plots.ylabel('Density')
  plots.legend()  ;  plots.show()
  return

## <span style="font-weight:bold;font-size:1.1em;color:blue">The Starbucks Tall Latte Index</span>

The Starbucks Tall Latte Index originated in January 2004, when <span style="font-weight:bold;font-style: italic">The Economist</span> created it as a lighthearted spin-off of its famous Big Mac Index. Because Starbucks stores were rapidly expanding across global major cities and maintained uniform ingredients, recipes, and quality standards, the beverage served as a good global benchmark.   

While The Economist created the original index in 2004, other publications and financial organizations periodically adapt or revive the metric. In 2017, <span style="font-weight:bold;font-style: italic">The Wall Street Journal</span> published a widely cited version comparing tall latte prices across 29 major world cities to highlight global currency misalignments. Financial media regularly update global rankings of Starbucks prices to analyze local wage disparities, tariffs, and real estate overhead across dozens of countries.

## <span style="font-weight:bold;font-size:1.1em;color:blue">Exammple 1: Lattes vs. Retail Rent</span>

In the table named **star** below, the price of a Starbucks Tall Latte in a city is compared to cost of renting prime retail space (measured in USD per square foot).

**Research Question.** Test at the 0.05 level whether there is a significant correlation between the price of a Starbucks Tall Latte and the cost of renting prime retail space in a city.

In [3]:
star.show(7)

Region,City,Starbucks Tall Latte (USD),Prime Retail Rent (Annual USD/sq ft),Economic Insight
US,"New York, NY",4.95,2000,Massive premium real estate skew vs. controlled product ...
US,"Los Angeles, CA",4.75,850,Rodeo Drive & local premium zones drive high real estate ...
US,"San Francisco, CA",4.85,650,High local labor mandates inflate the baseline latte pri ...
US,"Miami, FL",4.45,450,Rapid post-pandemic real estate growth outpaces standard ...
US,"Chicago, IL",4.35,420,High-density Michigan Avenue corridor commands steep pre ...
US,"Boston, MA",4.4,350,"Dense, historic retail spaces keep urban vacancy low."
US,"Honolulu, HI",4.95,320,Extreme supply isolation inflates both coffee beans and ...


### Data Checks

### Regression

### Reporting Out

## <span style="font-weight:bold;font-size:1.1em;color:blue">Exammple 2: Lattes vs. Craft Beer</span>

In the table named **star_beer** below, the price of a Starbucks Tall Latte in a city is compared to cost of an imported beer (proxy for craft beer).

**Research Question.** Test at the 0.05 level whether there is a significant correlation between the price of a Starbucks Tall Latte and the price of a craft beer in a community.

In [4]:
star_beer.show(7)

City/Community,Market Tier,Starbucks Tall Latte Index (USD),Imported Beer Price (USD)
"New York (Manhattan), NY",High Urban,5.16,9.08
"Los Angeles, CA",High Urban,5.02,9.1
"San Francisco, CA",High Urban,5.18,9.19
"San Jose, CA",High Urban,5.05,9.05
"Boston, MA",High Urban,5.02,8.97
"Washington, DC",High Urban,4.92,8.74
"Seattle, WA",High Urban,4.95,8.75


### Data Checks

### Regression

### Reporting Out

## <span style="font-weight:bold;font-size:1.1em;color:blue">Exammple 3: Apple Store App Rating vs. App Price</span>

In the table named **apps** below, the number of rating stars on average an app receives in the Apple store is compared to the price of the app.

**Research Question.** Test at the 0.05 level whether there is a significant correlation between the price of an app and how many stars it receives in the Apple store.

In [5]:
app_ids = make_array( 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35 )
prices = make_array( 0.00, 0.00, 0.00, 0.00, 0.99, 0.99, 0.99, 1.99, 1.99, 1.99, 2.99, 2.99, 2.99, 3.99, 3.99, 3.99, 4.99, 4.99, 4.99, 5.99, 5.99, 5.99, 6.99, 6.99, 7.99, 7.99, 8.99, 8.99, 9.99, 9.99, 11.99, 12.99, 14.99, 19.99, 24.99)
ratings = make_array( 4.8, 4.7, 4.6, 4.5, 4.6, 4.4, 4.3, 4.5, 4.2, 4.1, 4.4, 4.3, 4.0, 4.2, 4.1, 3.9, 4.3, 4.0, 3.8, 4.1, 3.9, 3.7, 4.0, 3.8, 3.9, 3.7, 3.8, 3.6, 3.9, 3.5, 3.7, 3.6, 3.4, 3.5, 3.2 )
apps = Table().with_columns( 'App_ID', app_ids, 'Price_USD', prices, 'Rating_Stars', ratings )
apps.show(8)

App_ID,Price_USD,Rating_Stars
1,0,4.8
2,0,4.7
3,0,4.6
4,0,4.5
5,0.99,4.6
6,0.99,4.4
7,0.99,4.3
8,1.99,4.5


### Data Checks

### Regression

### Reporting Out